# Tea Disease Detection Model — ResNet50 Transfer Learning
**Tea Plantation Smart Monitoring System**

This notebook fine-tunes a ResNet50 backbone to classify tea leaf disease images into 4 categories:
- `healthy`
- `tea_blister_blight`
- `stem_branch_canker`
- `nutrient_deficiency`

**Input image size:** 416 × 416 × 3

**Artifact saved:** `disease_resnet50.keras`

---
### Dataset directory structure expected:
```
data/disease/
├── healthy/                  ← place healthy leaf images here
├── tea_blister_blight/       ← place blight images here
├── stem_branch_canker/       ← place canker images here
└── nutrient_deficiency/      ← place deficiency images here
```
Minimum recommended: **200 images per class** (800 total).


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {bool(tf.config.list_physical_devices("GPU"))}')

# ─── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_DIR      = os.path.join(BASE_DIR, 'data', 'disease')
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'tea-plantation-system', 'ml_training', 'model_artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

CLASS_NAMES   = ['healthy', 'tea_blister_blight', 'stem_branch_canker', 'nutrient_deficiency']
IMG_SIZE      = (416, 416)
BATCH_SIZE    = 16
NUM_CLASSES   = 4
EPOCHS_FROZEN = 50
EPOCHS_FINETUNE = 20

print(f'Data directory : {DATA_DIR}')
print(f'Artifacts dir  : {ARTIFACTS_DIR}')


## 1. Dataset Setup & ImageDataGenerator
Create the class subdirectories if they don't exist, then set up augmentation pipelines.


In [ ]:
# Create placeholder directories so the generator doesn't crash on first run
for cls in CLASS_NAMES:
    cls_path = os.path.join(DATA_DIR, cls)
    os.makedirs(cls_path, exist_ok=True)
    n_imgs = len([f for f in os.listdir(cls_path)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'  {cls:<25}: {n_imgs} images')

# Check total image count
total_images = sum(
    len([f for f in os.listdir(os.path.join(DATA_DIR, cls))
         if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    for cls in CLASS_NAMES
)
print(f'\nTotal images found: {total_images}')

if total_images == 0:
    print('\n[WARNING] No images found. Add images to the class directories above,')
    print('          then re-run from this cell. Training will be skipped below.')
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False


## 2. Augmentation Configuration


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2
)

if not SKIP_TRAINING:
    train_generator = train_datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        shuffle=True,
        seed=42,
        classes=CLASS_NAMES
    )

    val_generator = val_datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        shuffle=False,
        seed=42,
        classes=CLASS_NAMES
    )

    print(f'Training batches  : {len(train_generator)}')
    print(f'Validation batches: {len(val_generator)}')
    print(f'Class indices     : {train_generator.class_indices}')
else:
    print('SKIP_TRAINING=True — generators not created.')


## 3. Build ResNet50 Model
- Frozen ResNet50 backbone (ImageNet weights)
- Custom head: `GlobalAveragePooling2D → Dense(256, relu) → Dropout(0.3) → Dense(4, softmax)`


In [ ]:
def build_model(input_shape=(416, 416, 3), num_classes=4, trainable_base=False):
    base = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base.trainable = trainable_base

    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='tea_disease_resnet50')
    return model, base

model, base_model = build_model(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
model.summary(line_length=100)
print(f'\nTotal params:     {model.count_params():,}')
print(f'Trainable params: {sum(np.prod(w.shape) for w in model.trainable_weights):,}')


## 4. Phase 1 — Train Head (frozen backbone, 50 epochs)


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint(
        filepath=os.path.join(ARTIFACTS_DIR, 'disease_resnet50_phase1_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    )
]

if not SKIP_TRAINING:
    history1 = model.fit(
        train_generator,
        epochs=EPOCHS_FROZEN,
        validation_data=val_generator,
        callbacks=callbacks_phase1,
        verbose=1
    )

    # Plot phase 1 curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history1.history['accuracy'],     label='Train')
    axes[0].plot(history1.history['val_accuracy'], label='Val')
    axes[0].set_title('Phase 1 — Accuracy'); axes[0].legend(); axes[0].set_xlabel('Epoch')
    axes[1].plot(history1.history['loss'],     label='Train')
    axes[1].plot(history1.history['val_loss'], label='Val')
    axes[1].set_title('Phase 1 — Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')
    plt.suptitle('Phase 1 Training (Frozen Backbone)', fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print('Skipped: no images available.')


## 5. Phase 2 — Fine-tune (unfreeze last 30 layers, 20 epochs, lr=0.0001)


In [ ]:
if not SKIP_TRAINING:
    # Unfreeze the last 30 layers of the ResNet50 backbone
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    trainable_count = sum(np.prod(w.shape) for w in model.trainable_weights)
    print(f'Trainable params after unfreeze: {trainable_count:,}')

    model.compile(
        optimizer=Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    callbacks_phase2 = [
        EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ModelCheckpoint(
            filepath=os.path.join(ARTIFACTS_DIR, 'disease_resnet50_phase2_best.keras'),
            monitor='val_accuracy', save_best_only=True, verbose=1
        )
    ]

    history2 = model.fit(
        train_generator,
        epochs=EPOCHS_FINETUNE,
        validation_data=val_generator,
        callbacks=callbacks_phase2,
        verbose=1
    )

    # Plot phase 2 curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history2.history['accuracy'],     label='Train')
    axes[0].plot(history2.history['val_accuracy'], label='Val')
    axes[0].set_title('Phase 2 — Accuracy'); axes[0].legend(); axes[0].set_xlabel('Epoch')
    axes[1].plot(history2.history['loss'],     label='Train')
    axes[1].plot(history2.history['val_loss'], label='Val')
    axes[1].set_title('Phase 2 — Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')
    plt.suptitle('Phase 2 Fine-tuning (Last 30 Layers)', fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print('Skipped: no images available.')


## 6. Evaluation — Confusion Matrix & Classification Report


In [ ]:
if not SKIP_TRAINING:
    val_generator.reset()
    y_pred_probs = model.predict(val_generator, verbose=1)
    y_pred  = np.argmax(y_pred_probs, axis=1)
    y_true  = val_generator.classes

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title('Confusion Matrix — Disease ResNet50', fontsize=14)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

    print('\n=== Classification Report ===')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # Per-class accuracy
    per_class = cm.diagonal() / cm.sum(axis=1)
    for cls, acc in zip(CLASS_NAMES, per_class):
        print(f'  {cls:<25}: {acc:.4f}')
else:
    print('Skipped: no images available.')


## 7. Save Final Model


In [ ]:
if not SKIP_TRAINING:
    save_path = os.path.join(ARTIFACTS_DIR, 'disease_resnet50.keras')
    model.save(save_path)
    print(f'Model saved: {save_path}')

    # Quick load verification
    loaded = keras.models.load_model(save_path)
    print(f'Verification load OK — output shape: {loaded.output_shape}')
else:
    print('No model to save: SKIP_TRAINING=True.')

print('\nDisease model notebook complete.')
